# DX 704 Week 12 Project

This week's project will revisit the email spam classifier project from week 9 using large language model embeddings instead of custom features.


The full project description and a template notebook are available on GitHub: [Project 12 Materials](https://github.com/bu-cds-dx704/dx704-project-12).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Imports

In [2]:
import pandas as pd
import numpy as np
import re
import json
import sklearn.linear_model
import torch
import time

from tqdm import tqdm
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression

# You may need to install torch and transformers.
# Google Colab will have these installed already.
#
# %pip install transformers torch --upgrade

from transformers import AutoTokenizer, AutoModel, BertTokenizer, BertModel

/Users/arunram/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NotFoundError: dlopen(/Users/arunram/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tensorflow-plugins/libmetal_plugin.dylib, 0x0006): Library not loaded: @rpath/_pywrap_tensorflow_internal.so
  Referenced from: <8B62586B-B082-3113-93AB-FD766A9960AE> /Users/arunram/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tensorflow-plugins/libmetal_plugin.dylib
  Reason: tried: '/Users/arunram/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tensorflow-plugins/../_solib_darwin_arm64/_U@local_Uconfig_Utf_S_S_C_Upywrap_Utensorflow_Uinternal___Uexternal_Slocal_Uconfig_Utf/_pywrap_tensorflow_internal.so' (no such file), '/Users/arunram/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tensorflow-plugins/../_solib_darwin_arm64/_U@local_Uconfig_Utf_S_S_C_Upywrap_Utensorflow_Uinternal___Uexternal_Slocal_Uconfig_Utf/_pywrap_tensorflow_internal.so' (no such file), '/Users/arunram/.pyenv/versions/3.11.9/lib/_pywrap_tensorflow_internal.so' (no such file), '/System/Volumes/Preboot/Cryptexes/O

In [2]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


## Functions

In [16]:
def show_dict_slice(dict, divisor):
    portion_len = len(dict) // divisor
    output = dict(list(dict.items())[:portion_len])
    print(output)

In [15]:
# Combine Subject and Message with a separator for BERT
# The tokenizer will automatically add [CLS] at the start and [SEP] between segments.

def combine_text(row):
    # Use spaces to clearly separate subject and message text
    return row['Subject'] + " " + row['Message']

### process_and_extract_features

In [17]:

def process_and_extract_features(text: str) -> dict:
        """
        Cleans the input text and extracts word counts as features.

        The cleaning steps include:
        1. Converting text to lowercase.
        2. Removing punctuation (non-alphanumeric characters except spaces/newlines).
        3. Splitting the text into tokens (words).
        4. Removing common English stopwords (optional, but highly recommended for NB).

        Args:
            text (str): The combined subject and message text.

        Returns:
            dict: A dictionary where keys are words (features) and values are counts.
        """

        #  list of English stopwords (to be expanded)
        stopwords = set([
            'a', 'an', 'the', 'and', 'but', 'or', 'of', 'to', 'in', 'on',
            'is', 'it', 'for', 'with', 'at', 'by', 'as', 'do', 'he', 'she',
            'i', 'me', 'my', 'you', 'your', 'we', 'our', 'they', 'their',
            'this', 'that', 'was', 'were', 'be', 'are', 'not', 'have', 'had',
            'from', 'so', 'if', 'all', 'out', 'up', 'down', 'about', 'just'
        ])

        text    = str(text).lower()

        # Remove anything that is not a word character or space (i.e., punctuation)
        # Using  RegEx to replace non-word/non-space characters with a space
        text    = re.sub(r'[^\w\s]', ' ', text)

        # Tokenize (split by whitespace) and filter out short tokens and stopwords
        tokens  = []

        for word in text.split():
            if len(word) > 1 and word not in stopwords:
                tokens.append(word)


        # tokens  = [word for word in text.split() if len(word) > 1 and word not in stopwords]

        # Count token repetitions
        feature_counts = Counter(tokens)
        print(f"Extracting {len(feature_counts)} features from text of length {len(text)}")

        feature_counts_dict = dict(feature_counts)
        # print(f"Features: {list(feature_counts_dict.items())}\n")

        return feature_counts_dict

### train_naive_bayes_model

Calculates the smoothed likelihoods:$$\text{Likelihood: } P(f|C) = \frac{\text{Count}(f, C) + \alpha}{\sum_{f'} \text{Count}(f', C) + \alpha \cdot |\text{Vocabulary}|}$$These resulting values, $P(f|C)$, are small positive floating-point numbers (between 0 and 1).

In [3]:
# def train_naive_bayes_model(train_df: pd.DataFrame,
#                             feature_col: str    = 'features_dict',
#                             label_col: str      = 'Spam/Ham',
#                             alpha: int          = 1) -> pd.DataFrame:

#     """
#     Trains a Multinomial Naive Bayes model with Laplace smoothing (alpha=1)
#     and returns a DataFrame of feature likelihoods P(f|C).

#     Arguments:
#         train_df: Training DataFrame containing the specified feature and label columns.
#         feature_col: The name of the column containing the word count dictionary.
#         label_col: The name of the column containing the numeric class label (0=ham, 1=spam).
#         alpha: Smoothing parameter.

#     Returns:
#         DataFrame with columns: 'feature', 'ham_probability', 'spam_probability'.
#     """
#     print("Training Naive Bayes Model and generating df of feature probabilities...")

#     # ---  Aggregate Feature Counts ---
#     feature_counts_ham  = Counter()
#     feature_counts_spam = Counter()
#     total_counts_ham    = 0
#     total_counts_spam   = 0

#     for _, row in train_df.iterrows():
#         label       = row[label_col]
#         features    = row[feature_col]

#         for feature, count in features.items():
#             if label == 0: # Ham
#                 feature_counts_ham[feature] += count
#                 total_counts_ham            += count
#             else: # Spam
#                 feature_counts_spam[feature] += count
#                 total_counts_spam            += count

#     # ---  Determine Vocabulary ---
#     vocabulary = set(feature_counts_ham.keys()) | set(feature_counts_spam.keys())
#     vocab_size = len(vocabulary)

#     # ---  Calculate Likelihoods (P(f|C) with Laplace Smoothing) ---
#     likelihood_data = []

#     # Denominators: (Total word count in class + alpha * |Vocabulary|)
#     denominator_ham  = total_counts_ham + alpha * vocab_size
#     denominator_spam = total_counts_spam + alpha * vocab_size

#     for feature in vocabulary:
#         count_ham  = feature_counts_ham.get(feature, 0)
#         count_spam = feature_counts_spam.get(feature, 0)

#         # Calculate P(f|C)
#         prob_ham  = (count_ham + alpha) / denominator_ham
#         prob_spam = (count_spam + alpha) / denominator_spam

#         likelihood_data.append({
#             'feature':          feature,
#             'ham_probability':  prob_ham,
#             'spam_probability': prob_spam
#         })

#     return pd.DataFrame(likelihood_data)



### classify_document

In [7]:
def classify_document(document_features: dict, model: dict) -> tuple[float, float]:

    """
    Computes the normalized posterior probabilities that a document belongs to the 'Ham' or 'Spam' class
    using a Naive Bayes classifier with log-probabilities for numerical stability.

    Parameters:
        ----------
        document_features : dict
            A dictionary representing the features of the document, where keys are feature names (e.g., words or tokens)
            and values are their corresponding counts or frequencies in the document.

        model : dict
            A dictionary containing the trained Naive Bayes model parameters:
            - 'prior': A tuple or list with prior probabilities for the Ham and Spam classes, respectively.
            - 'likelihood': A dictionary mapping each feature to a tuple of likelihoods (P(feature|Ham), P(feature|Spam)).

    Returns:
        -------
        tuple[float, float]
            A tuple containing the normalized posterior probabilities:
            - P_Ham: Probability that the document is Ham.
            - P_Spam: Probability that the document is Spam.

        Notes:
        ------
        - The function uses log-space arithmetic to prevent numerical underflow when multiplying many small probabilities.
        - It applies the log-sum-exp trick to safely normalize the log-probabilities.
        - If the computed denominator is zero (which may happen due to extreme underflow), it returns equal probabilities (0.5, 0.5).
    """

    # print(f"Calculating predictions for: {document_features}")

    # Initialize log-priors
    log_numerator_ham  = np.log(model['prior'][0])
    log_numerator_spam = np.log(model['prior'][1])

    # Add weighted log-likelihoods: sum(count(f) * log(P(f|C)))
    for feature, count in document_features.items():
        if feature in model['likelihood']:
            log_prob_ham  = np.log(model['likelihood'][feature][0])
            log_prob_spam = np.log(model['likelihood'][feature][1])

            log_numerator_ham  += count * log_prob_ham
            log_numerator_spam += count * log_prob_spam

    # Log-sum-exp trick for safe normalization
    max_log = max(log_numerator_ham, log_numerator_spam)

    # Convert back to unnormalized probabilities
    num_ham  = np.exp(log_numerator_ham - max_log)
    num_spam = np.exp(log_numerator_spam - max_log)

    # Normalize
    denominator = num_ham + num_spam

    if denominator == 0:
        return (0.5, 0.5)

    P_Ham   = num_ham / denominator
    P_Spam  = num_spam / denominator

    return (P_Ham, P_Spam)

### calculate_roc_curve

In [8]:

# Assumes dataframe's true label = 'Spam/Ham' and that the predicted label = 'spam'

def calculate_roc_curve(df: pd.DataFrame, threshold: float) -> tuple[float, float]:
    """
    Calcs the True Positive Rate (TPR) and False Positive Rate (FPR)
    for a given threshold, where Spam (1) is the positive class.

    Args:
        df: DataFrame containing 'Spam/Ham' (True Labels) and 'spam' (Probabilities).
        threshold: the classification cutoff (predicted spam probability >= threshold --> predict Spam).

    Returns:
         tuple (FPR, TPR).
    """

    #  predictions based on  threshold
    predictions = (df['spam'] >= threshold).astype(int)

    # True labels
    y_true      = df['Spam/Ham']

    #  Confusion Matrix: [[TN, FP], [FN, TP]]
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels = [0, 1]).ravel()

    # Calculate True Positive Rate (TPR or Recall)
    total_positives = tp + fn
    tpr             = tp / total_positives if total_positives > 0 else 0.0 # division by zero

    # False Positive Rate (FPR)
    total_negatives = tn + fp
    fpr             = fp / total_negatives if total_negatives > 0 else 0.0

    return fpr, tpr


## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [ ]:
# !wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

# '/content/drive/MyDrive/DX704/dx704-project-09-main/enron_spam_data.csv'

In [4]:
# pandas can read the zip file directly
path            = 'https://raw.githubusercontent.com/MWiechmann/enron_spam_data/master/enron_spam_data.zip'
enron_spam_data = pd.read_csv(path, compression='zip')

# enron_spam_data = pd.read_csv("/Users/arunram/Desktop/Data Science/BU_MSDS/Module 8/dx704-project-09-main/enron_spam_data.csv")
enron_spam_data.head(10)

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
5,5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",ham,1999-12-14
6,6,meter 1517 - jan 1999,"george ,\ni need the following done :\njan 13\...",ham,1999-12-14
7,7,duns number changes,fyi\n- - - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
8,8,king ranch,there are two fields of gas that i am having d...,ham,1999-12-14
9,9,re : entex transistion,thanks so much for the memo . i would like to ...,ham,1999-12-14


In [5]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Download BERT Model

We will use a pre-trained BERT model to extract embedding vectors as described in lesson 2.1 this week.
Here is sample code to download the model from [Hugging Face](https://huggingface.co/) and extract one vector.
This model is small enough that you can run it with CPU only, but GPUs will be faster if available.

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

To download the pre-trained model from Hugging Face, you will need to sign up for a free account with them at https://huggingface.co/join.
Afterwards, get an API token and if you are using Google Colab, save it as a secret named HF_TOKEN.

In [12]:
MODEL_NAME = "bert-base-uncased"

tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model  = AutoModel.from_pretrained(MODEL_NAME)
bert_model.to(device)
bert_model.eval()


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [13]:
@torch.no_grad()
def embed_text(text):
    batch   = [text]
    inputs  = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    # CLS token embedding is the first token's hidden state
    cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: [batch_size, 768]
    return cls_emb.cpu()

In [14]:
v = embed_text("Hi, will you buy my spam?")
v.shape

torch.Size([1, 768])

## Part 3: Create Embedding Vectors

- Use BERT to create embeddings for each email in the Enron data set. You will have to decide how to combine the different columns of the original data set to produce one embedding vector.


- Save your embeddings in a file "`embeddings.tsv.gz`" with two columns, `Message ID` and `embedding_vector_json`, where embedding_vector_json is a JSON-encoded list.

- Make sure that embedding_vector_json is a 1-dimensional list, not 2-dimensional.

- Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.


Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [18]:
# YOUR CHANGES HERE
df            = enron_spam_data.copy()
df['Subject'] = df['Subject'].fillna('')
df['Message'] = df['Message'].fillna('')

In [19]:
df['Combined_Text'] = df.apply(combine_text, axis=1)
df['Combined_Text'].head()


0                        christmas tree farm pictures 
1    vastar resources , inc . gary , production fro...
2    calpine daily gas nomination - calpine daily g...
3    re : issue fyi - see note below - already done...
4    meter 7268 nov allocation fyi .\n- - - - - - -...
Name: Combined_Text, dtype: object

In [21]:

#  Load BERT Model and Tokenizer
# Using 'bert-base-uncased' as a standard, widely available pre-trained model.
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model     = BertModel.from_pretrained('bert-base-uncased')

# Use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Set model to evaluation mode

# 3. Generate Embeddings
embeddings = []

MAX_LENGTH = 512 # Set a max length, typical is 512 for BERT. Truncate long emails.

print("Generating BERT embeddings for each document in 'Combined_Text'...")
start = time.time()
for text in tqdm(df['Combined_Text']):
    try:
        # Tokenize the text
        inputs = tokenizer(
            text,
            return_tensors  = 'pt',
            truncation      = True,
            padding         = 'max_length',
            max_length      = MAX_LENGTH
        ).to(device)

        # Generate model output without calculating gradients
        with torch.no_grad():
            outputs = model(**inputs)

        # Extract the embedding vector:
        # The first vector in the last_hidden_state output corresponds to the [CLS] token.
        #  use this vector as the single embedding for the entire email text.
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

        embeddings.append(cls_embedding)

    except Exception as e:
        # Handle potential errors during processing (e.g., very large inputs, model issues)
        print(f"Error processing email: {e}. Appending a zero vector.")
        # If an error occurs, append a zero vector as a placeholder (size 768 for bert-base)
        embeddings.append(np.zeros(768))

# Convert list of embeddings (NumPy arrays) into  single NumPy array
embeddings_array  = np.stack(embeddings)
end               = time.time()
execution_time    = end - start


print(f"Execution time: {execution_time} seconds")
print("\nEmbedding Generation Complete.")
print(f"Shape of final embeddings array: {embeddings_array.shape}")
print(f"Dimensionality of each embedding vector: {embeddings_array.shape[1]}")

Generating BERT embeddings for each document in 'Combined_Text'...


100%|██████████| 33716/33716 [09:48<00:00, 57.33it/s]


Execution time: 588.1703226566315 seconds

Embedding Generation Complete.
Shape of final embeddings array: (33716, 768)
Dimensionality of each embedding vector: 768


In [22]:
message_ids = df['Message ID']

# Verify length matches Message IDs
if len(embeddings_array) != len(message_ids):
    raise ValueError(f"Length mismatch: {len(embeddings_array)} embeddings for {len(message_ids)} messages.")

In [23]:

# Convert each embedding vector (NumPy array row) to a JSON-encoded string (1D list)

digits                = 4
embedding_vector_json = [json.dumps([round(x, digits) for x in row.tolist()]) for row in embeddings_array]

In [24]:
df_embedded_vectors = pd.DataFrame({
    'Message ID':             message_ids,
    'embedding_vector_json':  embedding_vector_json
})

In [25]:
# YOUR CHANGES HERE

file_path = 'embeddings.tsv.gz'

df_embedded_vectors.to_csv(file_path, sep = '\t', index = False)
print(f"Saved {len(df_embedded_vectors)} paragraph vectors to {file_path}.")

Saved 33716 paragraph vectors to embeddings.tsv.gz.


Submit "embeddings.tsv.gz" in Gradescope.

## Part 4: Train a Linear Regression

- Train an ordinary least squares regression for spam/ham status where spam is treated as target value 1 and ham is treated as target value 0 with your embeddings above as the only input variables.

- Save the coefficients of your linear model in a file "`coefficients.tsv`" with columns `dim` and `coefficient` where dim specifies the offset in the embedding vector (0-767).


- Don't worry about the bias term (but your model should still have it).


Steps:Load Data and Embeddings: Load the original data and the (placeholder) embeddings.Prepare Target Variable (y): Create the binary target variable, where 'spam' is 1 and 'ham' is 0.Prepare Feature Matrix (X): Use the embeddings_array as the feature matrix $\mathbf{X}$.Train/Test Split: Split the data to properly evaluate model performance.Train Model: Train a Logistic Regression model.

In [8]:
embeddings_df = pd.read_csv('embeddings.tsv.gz', sep = '\t')
embeddings_df.head()

,Message ID,embedding_vector_json
0,0,"[-0.4015, 0.2949, -0.1067, -0.1419, 0.2905, -0..."
1,1,"[-0.6057, -0.0498, 0.1918, 0.1134, 0.3432, -0...."
2,2,"[-0.7509, -0.0914, -0.2913, -0.2519, -0.028, -..."
3,3,"[-0.7712, -0.2957, -0.2422, -0.2047, -0.4219, ..."
4,4,"[-0.7081, -0.1852, 0.2904, -0.2171, -0.4848, 0..."


In [11]:
embeddings_array = embeddings_df['embedding_vector_json'].apply(lambda x: np.array(json.loads(x))).tolist()
embeddings_array = np.stack(embeddings_array)
embeddings_array.shape

(33716, 768)

In [21]:
# YOUR CHANGES HERE


N = len(df)
D = embeddings_array.shape[1] # BERT base dimensionality
print(f"Number of samples N = {N}; Dimensionality D = {D}")

#  Target y =  mapping

y = df['Spam/Ham'].map({'spam': 1, 'ham': 0}).values

#  Feature Matrix
X = embeddings_array

Number of samples N = 33716; Dimensionality D = 768


In [27]:
# Train/Test Split

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size = 0.2, random_state = 42, stratify = y
# )

# print(f"\nTraining set size (X): {X_train.shape}")
# print(f"Test set size (X): {X_test.shape}")


In [ ]:

# Train Logistic Regression model as  linear classifier

# model = LogisticRegression(
#     solver        = 'liblinear',  # Good for smaller datasets
#     C             = 10,                 # Low regularization strength
#     random_state  = 42
# )


In [22]:

model = LinearRegression(
    fit_intercept = True
)

In [ ]:
# # print("\n LogReg training...")
# model.fit(X_train, y_train)

# # Evaluate Model
# y_pred    = model.predict(X_test)
# accuracy  = accuracy_score(y_test, y_pred)

# print("\n--- Model Evaluation on Test Set ---")
# print(f"Accuracy: {accuracy:.4f}")
# print("\nClassification Report:")

# Note: With placeholder zero embeddings, the model performance will be poor/random.
#
# print(classification_report(y_test, y_pred, target_names=['ham (0)', 'spam (1)']))

In [23]:
model.fit(X, y)

# Evaluate Model
y_pred    = model.predict(X)

In [24]:
coefficients  = model.coef_.flatten()
dim           = np.arange(len(coefficients))

# print(dim)
coefficients_df = pd.DataFrame({
    'dim': dim,
    'coefficient': coefficients
})

coefficients_df.head(10)

,dim,coefficient
0,0,-1.146779
1,1,-1.204849
2,2,-1.151071
3,3,-1.259390
4,4,-1.163666
5,5,-1.168466
6,6,-1.132924
7,7,-1.161412
8,8,-1.245692
9,9,-1.184708


In [25]:
# YOUR CHANGES HERE

file = 'coefficients.tsv'

coefficients_df.to_csv(file, sep = '\t', index = False)
print(f"Saved {len(coefficients_df)} coefficients to {file}.")

Saved 768 coefficients to coefficients.tsv.


Submit "coefficients.tsv" in Gradescope.

## Part 5: Search for Relevant Documents

The file "queries.tsv" specifies 10 queries. For each of the queries, encode them as a vector, and find the message that is closest using $L_2$ (Euclidean) distance between the query vector and every email embedding.

Save your results in a file "query_matches.tsv" with columns `query_id`, `query_vector_json`, and `Message ID`.

* Encode Queries: Use  BERT model to convert each query into a 768-dim vector.
* Calculate $L_2$ Distance: for each query vector, calculate Euclidean distance ($L_2$) to every email embedding.
* Find Closest Match: Identify the `Message ID` corresponding to the minimum L_2 distance.

In [32]:
# YOUR CHANGES HERE

path = '/content/drive/MyDrive/DX704/dx704-project-12-main/queries.tsv'
queries_df = pd.read_csv(path, sep = '\t')
queries_df

,query_id,query
0,1,accounting arrangements
1,2,sales higher than production
2,3,asked lawyer to write letter about unexpected ...
3,4,engineering exam
4,5,discounted tickets
5,6,unexecuted agreement
6,7,well bore
7,8,capacity problem
8,9,london partner
9,10,dormant account


In [33]:
queries   = queries_df['query'].tolist()
query_ids = queries_df['query_id'].tolist()

In [34]:
# load BERT Model and Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model     = BertModel.from_pretrained('bert-base-uncased')
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [35]:

query_vectors = []
MAX_LENGTH    = 512

print("\nEncoding 10 queries using BERT...")
for text in queries:
    # Tokenize and move input to device
    inputs = tokenizer(
        text,
        return_tensors  = 'pt',
        truncation      = True,
        padding         = 'max_length',
        max_length      = MAX_LENGTH
    ).to(device)

    # Generate model output and extract [CLS] embedding
    with torch.no_grad():
        outputs   = model(**inputs)
    cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    query_vectors.append(cls_embedding)

query_vectors_array = np.stack(query_vectors)


Encoding 10 queries using BERT...


In [36]:
from scipy.spatial.distance import cdist

# L2 / Euclidean Distance and Closest Match
#  cdist calculates the distance between every vector in X (queries) and every vector in Y (emails)
print("Calculating L2 distances (Euclidean Distance)...")

#  matrix of shape (10 queries, 33716 emails)
distance_matrix     = cdist(query_vectors_array,
                            embeddings_array,
                            metric = 'euclidean'
                            )

# index of minimum distance for each query
closest_indices     = np.argmin(distance_matrix, axis=1)

#  Message ID of closest match
matched_message_ids = message_ids[closest_indices]

print("converting query vectors to json lists ...")

# Convert query vectors to JSON-encoded lists
query_vector_json = [json.dumps(row.tolist()) for row in query_vectors_array]

Calculating L2 distances (Euclidean Distance)...
converting query vectors to json lists ...


In [37]:
query_matches_df = pd.DataFrame({
    'query_id':           query_ids,
    'query_vector_json':  query_vector_json,
    'Message ID':         matched_message_ids
})

In [38]:
# YOUR CHANGES HERE

query_matches_df.to_csv('query-matches.tsv', sep = '\t', index = False)
print(f"Saved {len(query_matches_df)} query matches to 'query_matches.tsv'.")

Saved 10 query matches to 'query_matches.tsv'.


Submit "query_matches.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.